In [1]:
import io
import os
import time
import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

#### Resources - Geosphere Austria
##### Weather station daily
    resource_id:     klima-v2-1d
    licence:    Creative Commons Attribution 4.0
    types:    station
    modes:    historical
    data portal link:    [link](https://data.hub.geosphere.at/dataset/klima-v2-1d)


In [ ]:
load_dotenv()

DB_CONFIG = {
    "host": os.environ["DB_HOST"],
    "port": int(os.getenv("DB_PORT", 5432)),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}

#ENDPOINT = "https://geosphere.at"
ENDPOINT = "https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1d"
MIN_SECONDS_PER_REQUEST = 15.1  # Strictly <= 240 requests/hour (3600 / 240 = 15.0s)
# users requests are limited to 5 requests per second and 240 requests per hour.
# See https://dataset.api.hub.geosphere.at/v1/docs/user-guide/request-limit.html
MAX_RETRIES = 3

TARGET_PARAMETERS = [
    "tl_ii", "tlmax", "vvbft_ii", "rfb_ii", "rf_ii", "rf_mittel",
    "rr_iii", "sh", "p_mittel", "rfb_mittel", "tl_mittel", "vv_mittel",
]
# request size limit is enforced. When the request format is JSON or CSV the limit is 1000000 values
# The request size is calculated before the request is processed.
# Therefore, a request may hit the request size limit even if no values for the requested range exist.
# the request for 12 parameters for a 24-hour interval on a 6 * 8 km grid results in a request size of 12*24*(6*8)=13824

# The request response header datapoint-count returns the request size in number of requested datapoints.


In [5]:
# Convert pandas NaN/None to Python None for Postgres NULL
def clean_val(val, cast_type=float):    
    if pd.isna(val) or val == "":
        return None
    return cast_type(val)

In [ ]:
# Load pending (or interrupted) rows from public.wildfire_station_dates
def fetch_pending_tasks(conn) -> pd.DataFrame:
    query = """
        SELECT id, station_id, before_date
        FROM public.wildfire_station_dates
        WHERE status IN ('pending', 'processing')
           OR status IS NULL
        ORDER BY station_id, before_date;
    """
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=["id", "station_id", "before_date"])

# Fetch valid station_ids to prevent FK violations on optional substation
def fetch_valid_station_ids(conn) -> set:
    with conn.cursor() as cur:
        cur.execute("SELECT station_id FROM public.klima_stations;")
        return {row[0] for row in cur.fetchall()}

# Deduplicate (station_id, before_date) and group consecutive dates into blocks
def build_contiguous_blocks(df: pd.DataFrame) -> pd.DataFrame:
    df["date_obj"] = pd.to_datetime(df["before_date"])
    unique_dates = (
        df[["station_id", "date_obj", "before_date"]]
        .drop_duplicates()
        .sort_values(by=["station_id", "date_obj"])
        .reset_index(drop=True)
    )
    unique_dates["date_diff"] = unique_dates.groupby("station_id")["date_obj"].diff().dt.days
    unique_dates["block_id"] = (
        unique_dates["date_diff"].isna() | (unique_dates["date_diff"] > 1)
    ).cumsum()

    blocks = (
        unique_dates.groupby(["station_id", "block_id"])
        .agg(
            start_date=("date_obj", "min"),
            end_date=("date_obj", "max"),
            days_count=("date_obj", "count"),
        )
        .reset_index()
    )
    return blocks


def call_geosphere_api(station_id: int, start_date: pd.Timestamp, end_date: pd.Timestamp) -> str | None:
    """Call GeoSphere API with retry and backoff logic."""
    params = {
        "parameters": TARGET_PARAMETERS,
        "start": start_date.strftime("%Y-%m-%dT00:00:00"),
        "end": end_date.strftime("%Y-%m-%dT00:00:00"),
        "station_ids": str(station_id),
        "output_format": "csv",
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(ENDPOINT, params=params, timeout=30)

            if response.status_code == 200:
                raw_text = response.text.strip()
                if raw_text.startswith("<"):
                    raise ValueError("Received HTML response instead of CSV.")
                return raw_text

            if response.status_code == 429:
                wait_sec = 60 * attempt
                print(f"<<<!!!>>> HTTP 429 Rate Limit hit. Waiting {wait_sec}s (attempt {attempt}/{MAX_RETRIES})...")
                time.sleep(wait_sec)
                continue

            if response.status_code >= 500:
                wait_sec = 15 * attempt
                print(f"<<<!!!>>> HTTP {response.status_code} Server Error. Retrying in {wait_sec}s...")
                time.sleep(wait_sec)
                continue

            raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")

        except requests.RequestException as exc:
            if attempt == MAX_RETRIES:
                raise
            wait_sec = 15 * attempt
            print(f"<<<!!!>>> Network error ({exc}). Retrying in {wait_sec}s...")
            time.sleep(wait_sec)

    return None


def run_enrichment():
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        pending_df = fetch_pending_tasks(conn)
        if pending_df.empty:
            print("[+] No pending rows found in public.wildfire_station_dates. Everything is up to date.")
            return

        valid_stations = fetch_valid_station_ids(conn)
        blocks = build_contiguous_blocks(pending_df)

        print(
            f"[+] Loaded {len(pending_df)} pending rows -> bundled into "
            f"{len(blocks)} contiguous API requests (max 240 req/hr)."
        )

        upsert_klima_sql = """
            INSERT INTO public.klima_data (
                obs_date, "time", station, substation,
                tl_ii, tlmax, tl_mittel,
                vvbft_ii, vv_mittel,
                rfb_ii, rfb_mittel, rf_ii, rf_mittel,
                rr_iii, sh, p_mittel
            )
            VALUES %s
            ON CONFLICT (station, obs_date) DO UPDATE SET
                "time"     = EXCLUDED."time",
                substation = EXCLUDED.substation,
                tl_ii      = EXCLUDED.tl_ii,
                tlmax      = EXCLUDED.tlmax,
                tl_mittel  = EXCLUDED.tl_mittel,
                vvbft_ii   = EXCLUDED.vvbft_ii,
                vv_mittel  = EXCLUDED.vv_mittel,
                rfb_ii     = EXCLUDED.rfb_ii,
                rfb_mittel = EXCLUDED.rfb_mittel,
                rf_ii      = EXCLUDED.rf_ii,
                rf_mittel  = EXCLUDED.rf_mittel,
                rr_iii     = EXCLUDED.rr_iii,
                sh         = EXCLUDED.sh,
                p_mittel   = EXCLUDED.p_mittel;
        """

        update_status_sql = """
            UPDATE public.wildfire_station_dates
            SET status = %s
            WHERE station_id = %s
              AND before_date BETWEEN %s AND %s
              AND status IN ('pending', 'processing');
        """

        for idx, block in blocks.iterrows():
            req_start_ts = time.monotonic()

            st_id = int(block["station_id"])
            start_dt = block["start_date"]
            end_dt = block["end_date"]
            start_date_py = start_dt.date()
            end_date_py = end_dt.date()

            print(
                f"[{idx + 1}/{len(blocks)}] Station {st_id}: "
                f"{start_date_py} to {end_date_py} ({block['days_count']} days)..."
            )

            # Mark block as 'processing'
            with conn:
                with conn.cursor() as cur:
                    cur.execute(update_status_sql, ("processing", st_id, start_date_py, end_date_py))

            try:
                raw_csv = call_geosphere_api(st_id, start_dt, end_dt)
                block_df = pd.read_csv(io.StringIO(raw_csv)) if raw_csv else pd.DataFrame()

                if block_df.empty:
                    print(f"  [-] Empty dataset returned for Station {st_id}. Marking as 'empty'.")
                    with conn:
                        with conn.cursor() as cur:
                            cur.execute(update_status_sql, ("empty", st_id, start_date_py, end_date_py))
                else:
                    block_df["obs_date"] = pd.to_datetime(block_df["time"]).dt.date

                    records = []
                    for _, row in block_df.iterrows():
                        raw_sub = clean_val(row.get("substation"), int)
                        # Only keep substation if it exists in public.klima_stations to avoid FK error
                        sub_id = raw_sub if raw_sub in valid_stations else None

                        records.append((
                            row["obs_date"],
                            str(row["time"]),
                            st_id,
                            sub_id,
                            clean_val(row.get("tl_ii"), float),
                            clean_val(row.get("tlmax"), float),
                            clean_val(row.get("tl_mittel"), float),
                            clean_val(row.get("vvbft_ii"), int),
                            clean_val(row.get("vv_mittel"), float),
                            clean_val(row.get("rfb_ii"), float),
                            clean_val(row.get("rfb_mittel"), float),
                            clean_val(row.get("rf_ii"), float),
                            clean_val(row.get("rf_mittel"), float),
                            clean_val(row.get("rr_iii"), float),
                            clean_val(row.get("sh"), float),
                            clean_val(row.get("p_mittel"), float),
                        ))

                    # Atomic transaction: insert klima_data + mark queue rows 'completed'
                    with conn:
                        with conn.cursor() as cur:
                            execute_values(cur, upsert_klima_sql, records)
                            cur.execute(update_status_sql, ("completed", st_id, start_date_py, end_date_py))

                    print(f"  Saved {len(records)} daily records to public.klima_data & marked 'completed'")

            except Exception as err:
                print(f"  !!! Failed Station {st_id} ({start_date_py}..{end_date_py}): {err}")
                with conn:
                    with conn.cursor() as cur:
                        cur.execute(update_status_sql, ("failed", st_id, start_date_py, end_date_py))

            # Enforce <= 240 requests/hour
            elapsed = time.monotonic() - req_start_ts
            sleep_for = MIN_SECONDS_PER_REQUEST - elapsed
            if sleep_for > 0 and idx < len(blocks) - 1:
                time.sleep(sleep_for)

        print("\n<<< Finished >>>")
    finally:
        conn.close()



In [ ]:
run_enrichment()

[+] Loaded 93450 pending rows -> bundled into 5016 contiguous API requests (max 240 req/hr).
[1/5016] Station 1: 2002-06-07 to 2002-06-20 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[2/5016] Station 1: 2003-09-03 to 2003-09-16 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[3/5016] Station 1: 2004-04-16 to 2004-04-29 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[4/5016] Station 1: 2004-08-05 to 2004-08-18 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[5/5016] Station 1: 2005-06-14 to 2005-06-27 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[6/5016] Station 1: 2008-08-26 to 2008-09-08 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[7/5016] Station 1: 2010-07-01 to 2010-07-14 (14 days)...
  Saved 14 daily records to public.klima_data & marked 'completed'
[8/5016] Station 1: 2011-08-24 t